In [4]:
#Log in to huggingface hub
from huggingface_hub import login
import os
from dotenv import load_dotenv
load_dotenv()
login(token = os.getenv("HFToken"))
#HFToken = [your token] in .env

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id,padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda:0",dtype=torch.bfloat16)


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [108]:
from datasets import load_dataset
ds = load_dataset("openai/gsm8k", "main")
ds["train"]["question"]

Column(['Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?', 'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?', 'Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?', 'James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?', ...])

In [113]:
prompt = [[
    {
        "role": "system",
        "content": "You are answering questions as short as possible. Show all steps."
    },
    {
        "role": "user",
        "content": question
    },
]for question in ds["train"]["question"][0:5]]
input = tokenizer.apply_chat_template(
    prompt,
    tokenize = True,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt",
    clean_up_tokenization_spaces=False
).to("cuda:0")
input_length = input["input_ids"].shape[1]

In [118]:
output = model.generate(**input,
                        max_new_tokens= 100,
                        do_sample=True,
                        temperature=0.2,
                        top_p = 0.95,
                        repetition_penalty=1.2,
                        )
for i in range(len(prompt)):
    print(tokenizer.decode(output[i][input_length:],skip_special_tokens=True),"\n")

In April, Natalia sold 48 clips.
May sales were half that amount (48 / 2 = 24).
Total sales for the two months is 48 + 24 = 72.

The answer is 72. 

To find out how much Weng earned, we need to convert the time from minutes to hours.

There are 60 minutes in an hour.
So, 50 minutes is equal to:
50 / 60 = 5/6 hours

Now, multiply her hourly wage by the number of hours worked:

$12/hour * (5/6) hours = $10

She earned $10 yesterday. 

1. Find out how much money Betty already has.
   - She has half of what's needed ($100 / 2 = $50).

2. Calculate how much money Betty got from her parents.
   - Parents gave her $15.

3. Calculate how much money Betty received from her grandparents.
   - Grandparents gave her twice as much as her parents (twice $15 = $30).

4. Add up everything Betty now has:
   - Money she had initially: $50 

1. Find out how many pages Julie has left after reading 12 pages.
   Remaining_pages = Total_pages - Pages_read_yesterday 
                   = 120 - 12
            